# nested-param-group-loop — ex2: manual SGD-with-weight-decay using per-group hparams

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `nested-param-group-loop`. Running the final beacon cell reports progress against the `Config: nested param-group loop` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: nested param-group loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nested-param-group-loop`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nested-param-group-loop"
DD_SUBTOPIC = "Config: nested param-group loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Manual SGD step with weight decay — per-group hparams

Ex1 read ONE per-group hparam (`lr`) inside the nested loop. The natural deepening: also read `weight_decay`. Weight decay adds a `-lr * wd * p` term to every parameter (or equivalently, an `L2` penalty on `p` in the loss):

```python
for group in optimizer.param_groups:
    lr = group['lr']
    wd = group.get('weight_decay', 0.0)
    for p in group['params']:
        if p.grad is None:
            continue
        g = p.grad
        if wd != 0.0:
            g = g + wd * p.data       # decoupled decay term
        p.data.add_(g, alpha=-lr)
```

**Why `group.get('weight_decay', 0.0)`.** Not every group has weight_decay set (some groups, e.g. biases / LayerNorm params, opt out). `.get(..., 0.0)` is the standard 'missing means no decay' convention.

**Sign convention.** Decay is `+wd * p` ADDED to the grad — since the step subtracts `lr * grad`, that pulls `p` toward zero. Match the sign convention used in `torch.optim.SGD(weight_decay=...)`.

**Vanilla SGD weight-decay vs AdamW decoupled decay.** The form above is the SGD style: decay is folded into the grad before the step. AdamW decouples decay so it's applied as a separate term outside the moment estimates — same arithmetic, different timing.

### Exercise 2 — manual SGD-with-weight-decay using per-group hparams

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the nested `param_groups → params` loop to manually perform one SGD step that reads BOTH `lr` and `weight_decay` per group and folds the decay term into the gradient before the update.
> Keywords: param-groups, weight-decay, sgd, per-group-hparams
> ```

**KCs targeted:** `read-multiple-group-hparams`, `weight-decay-folded-into-grad`

Implement `ex2_manual_sgd_wd_step(optimizer)`.

Like ex1, but each group now ALSO has a `weight_decay` hparam. The step rule per parameter is:
```
g_effective = p.grad + wd * p.data       (only if wd != 0)
p.data <- p.data - lr * g_effective
```
Algorithm:
1. Outer loop over `optimizer.param_groups`.
2. Read `lr = group['lr']` and `wd = group.get('weight_decay', 0.0)`.
3. Inner loop over `group['params']`.
4. Skip `p.grad is None`.
5. If `wd != 0`, use `g = p.grad + wd * p.data`;   otherwise `g = p.grad`.
6. `p.data.add_(g, alpha=-lr)`.

Do NOT call `optimizer.step()`. Output: `None` (in-place mutation).

In [ ]:
def ex2_manual_sgd_wd_step(optimizer):
    """Manual SGD step with per-group weight decay."""
    raise NotImplementedError()


def _test_ex2():
    # === Single group, weight_decay=0 ===
    p = t.nn.Parameter(t.tensor([1.0, 2.0, 3.0]))
    opt = t.optim.SGD([p], lr=0.1, weight_decay=0.0)
    p.grad = t.tensor([10.0, 20.0, 30.0])
    ex2_manual_sgd_wd_step(opt)
    expected = t.tensor([1.0, 2.0, 3.0]) - 0.1 * t.tensor([10.0, 20.0, 30.0])
    assert t.allclose(p.detach(), expected, atol=1e-6), (
        f'wd=0 step wrong: got {p.detach()}, expected {expected}'
    )

    # === Single group, weight_decay=0.1 ===
    p = t.nn.Parameter(t.tensor([1.0, 2.0, 3.0]))
    opt = t.optim.SGD([p], lr=0.1, weight_decay=0.1)
    p.grad = t.tensor([1.0, 1.0, 1.0])
    ex2_manual_sgd_wd_step(opt)
    # g_eff = grad + wd * p = [1,1,1] + 0.1*[1,2,3] = [1.1, 1.2, 1.3]
    # new p = old p - lr * g_eff = [1,2,3] - 0.1*[1.1,1.2,1.3]
    expected = t.tensor([1.0, 2.0, 3.0]) - 0.1 * t.tensor([1.1, 1.2, 1.3])
    assert t.allclose(p.detach(), expected, atol=1e-6), (
        f'wd=0.1 step wrong: got {p.detach()}, expected {expected}'
    )

    # === Two groups with DIFFERENT weight decay ===
    p1 = t.nn.Parameter(t.tensor([2.0, 2.0]))
    p2 = t.nn.Parameter(t.tensor([2.0, 2.0]))
    opt = t.optim.SGD([
        {'params': [p1], 'lr': 0.1, 'weight_decay': 0.0},
        {'params': [p2], 'lr': 0.1, 'weight_decay': 0.5},
    ])
    p1.grad = t.zeros(2)                    # only decay drives p1
    p2.grad = t.zeros(2)                    # only decay drives p2
    ex2_manual_sgd_wd_step(opt)
    # p1: g_eff = 0 + 0 * p1 = 0 -> p1 unchanged.
    # p2: g_eff = 0 + 0.5 * p2 = [1, 1]; new p2 = [2,2] - 0.1*[1,1] = [1.9, 1.9].
    assert t.allclose(p1.detach(), t.tensor([2.0, 2.0]), atol=1e-6), (
        f'wd=0 group should not move: got {p1.detach()}'
    )
    assert t.allclose(p2.detach(), t.tensor([1.9, 1.9]), atol=1e-6), (
        f'wd=0.5 group wrong: got {p2.detach()}'
    )

    # === grad=None must be SKIPPED ===
    p_a = t.nn.Parameter(t.tensor([5.0]))
    p_b = t.nn.Parameter(t.tensor([5.0]))
    opt = t.optim.SGD([p_a, p_b], lr=0.1, weight_decay=0.2)
    p_a.grad = t.tensor([1.0])
    # p_b.grad left as None
    ex2_manual_sgd_wd_step(opt)
    # p_a: g_eff = 1 + 0.2 * 5 = 2; new p_a = 5 - 0.1*2 = 4.8.
    assert t.allclose(p_a.detach(), t.tensor([4.8]), atol=1e-6), p_a.detach()
    assert t.allclose(p_b.detach(), t.tensor([5.0]), atol=1e-6), (
        f'p_b grad=None should not move, got {p_b.detach()}'
    )

    # === group.get('weight_decay', 0.0) — groups without WD key default to 0 ===
    # Construct two groups, the second one without weight_decay specified.
    p_x = t.nn.Parameter(t.tensor([1.0, 1.0]))
    p_y = t.nn.Parameter(t.tensor([1.0, 1.0]))
    # Build manually so one group is missing 'weight_decay'.
    opt = t.optim.SGD([
        {'params': [p_x], 'lr': 0.1, 'weight_decay': 0.0},
        {'params': [p_y], 'lr': 0.1, 'weight_decay': 0.0},
    ])
    # Now simulate a group WITHOUT weight_decay key by popping it:
    del opt.param_groups[1]['weight_decay']
    p_x.grad = t.ones(2)
    p_y.grad = t.ones(2)
    ex2_manual_sgd_wd_step(opt)
    # Both groups should behave as wd=0.
    assert t.allclose(p_x.detach(), t.tensor([0.9, 0.9]), atol=1e-6), p_x.detach()
    assert t.allclose(p_y.detach(), t.tensor([0.9, 0.9]), atol=1e-6), (
        f'missing weight_decay key should default to 0; got {p_y.detach()}'
    )

    # === In-place — same id and storage ===
    p = t.nn.Parameter(t.zeros(4))
    orig_ptr = p.data_ptr()
    opt = t.optim.SGD([p], lr=0.1, weight_decay=0.01)
    p.grad = t.ones(4)
    ex2_manual_sgd_wd_step(opt)
    assert p.data_ptr() == orig_ptr, 'param storage was reallocated'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_manual_sgd_wd_step(optimizer):
    for group in optimizer.param_groups:
        lr = group['lr']
        wd = group.get('weight_decay', 0.0)
        for p in group['params']:
            if p.grad is None:
                continue
            g = p.grad
            if wd != 0.0:
                g = g + wd * p.data
            p.data.add_(g, alpha=-lr)
```

**`group.get('weight_decay', 0.0)` not `group['weight_decay']`.** Some groups (e.g. the no-decay bias group) intentionally omit `weight_decay`. The `.get` form gives 0 as the natural default and doesn't crash with KeyError.

**Why `g = p.grad + wd * p.data`, not `p.grad.add_(wd * p.data)`.** The latter mutates the user's gradient tensor — a side effect that breaks gradient accumulation and gradient-norm logging. Allocate a fresh `g`; the step's in-place mutation is the only side effect this function should have.

**Why fold decay into grad, not into the step.** Mathematically equivalent for vanilla SGD, but matches the structure of `torch.optim.SGD`'s own source — easier to read against the reference. AdamW takes the opposite choice (decoupled decay) for very different numerical reasons.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()